# Example: Tonight's Evening Desk Review

In this example, we run the 7pm desk-review meeting. Today's intraday tape and queue artifacts are sitting on disk: the tape was written by [`production_runner.jl`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#production_runner) at every 30-minute fire, the queue accumulated trades that hit at least one compliance gate, and the [Core ProductionEngine notebook](eCornell-AI-Finance-S4-Example-Core-ProductionEngine-May-2026.ipynb) produced both. We score the day, walk the queue trade-by-trade with each engine snapshot in front of us, and write the night's signed-decision artifacts.

> __Learning Objectives:__
>
> By the end of this example, you will be able to:
> * __Score the day from the intraday tape:__ Load `data/intraday-tape/tape-2026-05-11.jld2`, plot wealth and drawdown by 30-minute fire, and attribute P&L by hour. Read the regime and elasticity trace alongside to see whether the engine's view of the market shifted across the day.
> * __Walk the queue and adjudicate each item:__ Render each [`MyComplianceQueueItem`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyComplianceQueueItem) with its violations and engine snapshot, decide approve / reject / modify per row, and emit one [`MySignedDecision`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedDecision) per queue entry. Apply a deterministic adjudication policy so the demo is reproducible and Lou can audit the rationale on every row.
> * __Persist the night's signed decisions:__ Write the adjudicated decisions to `data/decisions/decisions-2026-05-11.jld2` so the next-morning cron can read what the desk approved before submitting to Alpaca paper. Round-trip the file back and assert per-field equality so the commit step has its own integrity check.

Let's open the tape.

___

## Setup, Data and Prerequisites

In [1]:
include("Include.jl");

### Constants


In [2]:
# Date of the tape we are reviewing tonight
REVIEW_DATE = Date(2026, 5, 11)
B0 = 10_000.0

# Adjudicator identity stamped on the night's signed decisions
ADJUDICATOR = "Maya Chen"

# Bar labels for plot ticks (matches the 13-bar regular session)
BAR_TIMES = ["09:30", "10:00", "10:30", "11:00", "11:30", "12:00", "12:30",
             "13:00", "13:30", "14:00", "14:30", "15:00", "15:30"]


13-element Vector{String}:
 "09:30"
 "10:00"
 "10:30"
 "11:00"
 "11:30"
 "12:00"
 "12:30"
 "13:00"
 "13:30"
 "14:00"
 "14:30"
 "15:00"
 "15:30"

### Implementations

The desk's policy for tonight's queue maps each gate-violation pattern to a default adjudication action:

* `:news_severity` &rarr; reject (defer until news flow clarifies direction).
* `:concentration_cap` &rarr; modify down to roughly half the proposed quantity so the post-trade weight respects the cap.
* `:turnover_limit` only (no per-trade gate) &rarr; approve (per-trade gates clear; queued only because the basket-level cap was hit).
* `:position_size_limit` &rarr; approve (acceptable size on a known holding).

The `adjudicate(item::`[`MyComplianceQueueItem`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyComplianceQueueItem)`)` helper below encodes that policy. Lifting it out of the Task 2 cell keeps the let block focused on the loop and lets Lou read the policy in one place.

In [ ]:
function adjudicate(item::MyComplianceQueueItem)
    gv = Set(item.gate_violations);
    if :news_severity in gv
        return (:reject, nothing,
            "News severity above threshold; defer until news flow clarifies direction.");
    elseif :concentration_cap in gv
        # Modify down to roughly half the original quantity to respect the cap.
        return (:modify, max(1, div(item.qty, 2)),
            "Original size breaches concentration cap; sign at half-size to respect cap.");
    elseif :turnover_limit in gv && length(gv) == 1
        return (:approve, nothing,
            "Per-trade gates clear; queued only because basket-level turnover hit the cap.");
    elseif :position_size_limit in gv
        return (:approve, nothing,
            "Position size acceptable on a known holding; sign as proposed.");
    else
        return (:approve, nothing, "No adverse signal in violations; sign as proposed.");
    end
end

Load tonight's tape and queue artifacts off disk. Both are produced by the cron-driven [`production_runner.jl`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#production_runner); the [Core ProductionEngine notebook](eCornell-AI-Finance-S4-Example-Core-ProductionEngine-May-2026.ipynb) also writes the same files when run on the synthetic fixture day, so this notebook works whether the cron is live or we are running off the replay.

If today's queue is empty (no flagged trades), we mock four representative items so the adjudication walk-through has something to work on. In the code block below, we return `tape::Vector{NamedTuple}` (one entry per intraday fire) and `queue::Vector{`[`MyComplianceQueueItem`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyComplianceQueueItem)`}`.

In [ ]:
tape, queue = let
    # --- Step 1: Load today's intraday tape (one entry per fire) ---
    tape_path = joinpath(_PATH_TO_TAPE, "tape-$(Dates.format(REVIEW_DATE, "yyyy-mm-dd")).jld2");
    isfile(tape_path) || error("Tape not found at $(tape_path). Run the Core ProductionEngine notebook first.");
    tape_data = load_results(tape_path);
    tape_entries = tape_data["entries"]::Vector;

    # --- Step 2: Load today's queue (or mock four items if it's empty) ---
    queue_path = joinpath(_PATH_TO_QUEUE, "queue-$(Dates.format(REVIEW_DATE, "yyyy-mm-dd")).jld2");
    queue_items = isfile(queue_path) ? load_queue(queue_path) : MyComplianceQueueItem[];

    if length(queue_items) < 2
        # Replay-mode fallback: synthesize a representative queue across all four
        # gate categories so the adjudication walkthrough has signal.
        original_n = length(queue_items);
        mk_ts(h, m) = DateTime(REVIEW_DATE) + Hour(h) + Minute(m);
        queue_items = MyComplianceQueueItem[
            build(MyComplianceQueueItem, (
                id = "2026-05-11T11:30-MSFT-sell-1", timestamp = mk_ts(11, 30),
                ticker = "MSFT", qty = 4, side = :sell, proposed_weight = 0.05,
                gate_violations = [:news_severity],
                engine_snapshot = Dict{String,Any}("eta" => 4.7, "lambda_eff" => 0.07,
                    "sentiment" => -0.06, "regime" => "neutral", "news_severity" => 0.81))),
            build(MyComplianceQueueItem, (
                id = "2026-05-11T13:00-NVDA-buy-2", timestamp = mk_ts(13, 0),
                ticker = "NVDA", qty = 12, side = :buy, proposed_weight = 0.42,
                gate_violations = [:concentration_cap, :position_size_limit],
                engine_snapshot = Dict{String,Any}("eta" => 2.1, "lambda_eff" => 0.41,
                    "sentiment" => -0.12, "regime" => "bear"))),
            build(MyComplianceQueueItem, (
                id = "2026-05-11T14:30-AMZN-buy-3", timestamp = mk_ts(14, 30),
                ticker = "AMZN", qty = 6, side = :buy, proposed_weight = 0.22,
                gate_violations = [:turnover_limit],
                engine_snapshot = Dict{String,Any}("eta" => 1.8, "lambda_eff" => 0.10,
                    "sentiment" => -0.02, "regime" => "neutral"))),
            build(MyComplianceQueueItem, (
                id = "2026-05-11T15:30-AAPL-sell-4", timestamp = mk_ts(15, 30),
                ticker = "AAPL", qty = 9, side = :sell, proposed_weight = 0.18,
                gate_violations = [:position_size_limit],
                engine_snapshot = Dict{String,Any}("eta" => 2.0, "lambda_eff" => 0.05,
                    "sentiment" => 0.04, "regime" => "neutral"))),
        ];
        # Persist the synthetic queue so the rest of the loop has a clean artifact to read.
        save_queue!(queue_path, queue_items);
        println("Real queue had $(original_n) item(s); replaced with synthetic 4-item queue for demo.")
    end

    println("Loaded $(length(tape_entries)) tape entries from $(tape_path).")
    println("Loaded $(length(queue_items)) queue items from $(queue_path).")
    tape_entries, queue_items
end;

___
## Task 1: Score the Day from the Intraday Tape
In this task, we open the day's tape and produce the desk's standard scorecard: a wealth and drawdown plot by 30-minute fire, a per-hour P&L attribution, and a regime-and-η trace that shows whether the engine's view of the market shifted. This is the meeting's opening read: *what did today look like before we touch the queue?*

> __What should we see?__
>
> Wealth tracks SPY through the morning, dips at the shock bar (~11:00), and partially recovers; drawdown peaks an hour or two after the shock as the engine's reallocation locks in some of the loss. The regime classification stays at `:neutral` for all 13 bars because the EMA windows used by [`compute_lambda`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#compute_lambda) (21 and 63) cannot warm up enough to register a directional cross within a 13-bar buffer; this is the same warmup limitation we flagged in the [Core ProductionEngine notebook](eCornell-AI-Finance-S4-Example-Core-ProductionEngine-May-2026.ipynb), and the flat regime/η trace is itself a diagnostic the desk reads. With more days of buffer history (a real cron run rather than a single-day replay) the regime trace would resolve.

The cell-bound result is `scorecard::`[`DataFrame`](https://dataframes.juliadata.org/stable/lib/types/#DataFrames.DataFrame) summarizing per-hour wealth changes plus the displayed plots.

In [ ]:
scorecard = let
    n = length(tape);
    bar_idx = 1:n;

    # --- Step 1: Wealth and drawdown trace ---
    p1 = plot(bar_idx, [e.wealth / B0 for e in tape];
        lw = 2.5, c = :steelblue, marker = :circle, ms = 4,
        label = "W / W₀", ylabel = "Wealth (relative to W₀)",
        title = "Today's intraday tape: wealth and drawdown");
    p2 = plot(bar_idx, [100 * e.drawdown for e in tape];
        lw = 2.5, c = :firebrick, marker = :circle, ms = 4,
        label = "Drawdown", ylabel = "Drawdown (%)",
        xlabel = "Bar (ET)", xticks = (bar_idx, BAR_TIMES));
    plot!(p1, bg = "gray95", framestyle = :box, fg_legend = :transparent);
    plot!(p2, bg = "gray95", framestyle = :box, fg_legend = :transparent);
    display(plot(p1, p2; layout = grid(2, 1, heights = [0.55, 0.45]),
        size = (820, 520), left_margin = 8Plots.mm));

    # --- Step 2: Regime + η trace ---
    regime_codes = [e.regime == :bear ? 1 : (e.regime == :neutral ? 2 : 3) for e in tape];
    p3 = plot(bar_idx, regime_codes;
        lw = 2.5, c = :purple, marker = :diamond, ms = 5,
        label = "Regime", ylabel = "Regime", yticks = ([1, 2, 3], ["bear", "neutral", "bull"]),
        title = "Regime and elasticity through the day");
    p4 = plot(bar_idx, [e.eta for e in tape];
        lw = 2.5, c = :darkorange, marker = :circle, ms = 4,
        label = "η (CES elasticity)", ylabel = "η",
        xlabel = "Bar (ET)", xticks = (bar_idx, BAR_TIMES));
    plot!(p3, bg = "gray95", framestyle = :box, fg_legend = :transparent);
    plot!(p4, bg = "gray95", framestyle = :box, fg_legend = :transparent);
    display(plot(p3, p4; layout = grid(2, 1, heights = [0.5, 0.5]),
        size = (820, 460), left_margin = 8Plots.mm));

    # --- Step 3: Per-hour P&L attribution scorecard ---
    hour_buckets = [Hour(e.fire_time).value for e in tape];
    unique_hours = sort(unique(hour_buckets));
    rows = NamedTuple[];
    for h in unique_hours
        idxs = findall(==(h), hour_buckets);
        w_in = idxs[1] == 1 ? B0 : tape[idxs[1] - 1].wealth;
        w_out = tape[idxs[end]].wealth;
        push!(rows, (
            Hour = "$(lpad(h, 2, '0')):00 ET",
            W_in = "\$$(round(w_in, digits = 0))",
            W_out = "\$$(round(w_out, digits = 0))",
            ΔW_pct = "$(round(100 * (w_out - w_in) / w_in, digits = 2))%",
            Auto = sum(e.auto_n for e in tape[idxs]),
            Queue = sum(e.queued_n for e in tape[idxs])));
    end
    df = DataFrame(rows);
    println("Per-hour scorecard (May 11):")
    pretty_table(df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact));
    df
end;

___
## Task 2: Walk the Queue and Adjudicate
In this task, we work each [`MyComplianceQueueItem`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MyComplianceQueueItem) the engine flagged today. For each row we render the timestamp, the proposed trade, the gate violations, and the engine snapshot at the fire (η, λ, regime, news severity if relevant), then record an action: `:approve` (sign as proposed), `:reject` (cancel), or `:modify` (sign at a different quantity). Every adjudication carries a free-form `notes` field that Lou will read on Tuesday morning. The action per row is computed by the `adjudicate` helper defined under Setup.

> __What should we see?__
>
> The MSFT news-flagged item gets rejected (not enough conviction in the news direction to act). The NVDA concentration-cap item gets modified down to a quantity that fits the cap. The AMZN turnover-driven item gets approved as-is (the only reason it queued was the basket-level cap). The AAPL position-size item gets approved (acceptable size on a known holding).

The cell-bound result is `signed_decisions::Vector{`[`MySignedDecision`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedDecision)`}` covering every queue item, plus a `pretty_table` summary.

In [ ]:
signed_decisions = let
    # --- Step 1: Build a MySignedDecision per queue item using the Setup-cell adjudicator ---
    signed_at = DateTime(REVIEW_DATE) + Hour(19) + Minute(15);
    decisions = MySignedDecision[];
    rows = NamedTuple[];
    for q in queue
        action, mod_qty, notes = adjudicate(q);
        d = build(MySignedDecision, (
            queue_id = q.id, action = action, modified_qty = mod_qty,
            notes = notes, signed_by = ADJUDICATOR, signed_at = signed_at));
        push!(decisions, d);
        push!(rows, (
            Time = Dates.format(q.timestamp, "HH:MM"),
            Ticker = q.ticker,
            Side = q.side == :buy ? "BUY" : "SELL",
            Qty = q.qty,
            Violations = join(string.(q.gate_violations), ", "),
            Action = string(action),
            Modified = mod_qty === nothing ? "" : string(mod_qty),
            Notes = notes));
    end

    # --- Step 2: Render the adjudication record ---
    println("Tonight's queue adjudication ($(length(decisions)) items, signed by $(ADJUDICATOR)):")
    println("  approve = ", count(d -> d.action == :approve, decisions),
        ", reject = ", count(d -> d.action == :reject, decisions),
        ", modify = ", count(d -> d.action == :modify, decisions));
    println();
    pretty_table(DataFrame(rows); backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact));
    decisions
end;

___
## Task 3: Persist the Signed Decisions
In this task, we write tonight's adjudications to disk via [`save_signed_decisions!`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#save_signed_decisions!). The next-morning cron (or, in this course, the [Core TomorrowsTicket notebook](eCornell-AI-Finance-S4-Example-Core-TomorrowsTicket-May-2026.ipynb)) loads this file to know which queued trades were approved, modified, or rejected. We also load it back for an integrity check: round-tripped decisions should equal what we just signed.

> __What should we see?__
>
> The file lands at `data/decisions/decisions-2026-05-11.jld2`. Reloading produces a vector of [`MySignedDecision`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session4/#MySignedDecision) records identical to what we wrote (same actions, same modified quantities, same notes, same signer).

The cell-bound result is `decisions_path::String` (so the next notebook can pick the file up) and a console assertion that the roundtrip matches.

In [ ]:
decisions_path = let
    # --- Step 1: Persist the signed decisions ---
    path = joinpath(_PATH_TO_DECISIONS, "decisions-$(Dates.format(REVIEW_DATE, "yyyy-mm-dd")).jld2");
    save_signed_decisions!(path, signed_decisions);
    println("Wrote $(length(signed_decisions)) decisions to $(path).")

    # --- Step 2: Reload and verify the roundtrip ---
    reloaded = load_signed_decisions(path);
    @assert length(reloaded) == length(signed_decisions);
    for (a, b) in zip(reloaded, signed_decisions)
        @assert a.queue_id == b.queue_id;
        @assert a.action == b.action;
        @assert a.modified_qty == b.modified_qty;
        @assert a.notes == b.notes;
        @assert a.signed_by == b.signed_by;
        @assert a.signed_at == b.signed_at;
    end
    println("Roundtrip OK: $(length(reloaded)) decisions match what we signed.")
    path
end;

___
## Summary
This example loaded today's intraday tape and queue, scored the day on wealth, drawdown, regime, and elasticity, walked each queue item with the engine snapshot in front of us, and persisted the night's signed decisions for tomorrow's cron to consume. The signed file lives at `data/decisions/decisions-2026-05-11.jld2`.

> __Key Takeaways:__
>
> * __The tape is the desk's daily P&L lens:__ One audit entry per fire is enough to attribute returns by hour, locate the bar where the regime classification shifted, and trace how η responded. The picture matches the textbook senior-PM end-of-day review.
> * __Queue adjudication is rule-driven, not vibes-driven:__ Every action carries a written rationale that maps to the specific gate violations on the row. A signed decision with no notes is a finding for Lou's audit, not a clean approval.
> * __Signed decisions are the commit step:__ The next-morning cron reads `decisions-YYYY-MM-DD.jld2` and submits exactly what was signed (approving as-proposed, modifying to the recorded quantity, or skipping rejected items). The desk's evening sign-off is mechanically connected to the next morning's market open.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. Real production deployment requires regulatory approval, compliance review, and operational controls beyond what this example demonstrates.

___